Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 우선순위 및 Flex 추론 티어

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Inference_tiers.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

Gemini API는 비용과 지연 시간을 관리하는 데 도움이 되는 다양한 `service_tier`를 제공합니다.

**우선순위(Priority)** 및 **Flex** 티어는 표준 동기 엔드포인트를 사용하여 백그라운드 작업을 Flex로, 인터랙티브 작업을 Priority로 라우팅할 수 있습니다. 이를 통해 비동기 작업 관리의 복잡성을 제거하면서 특수 티어의 경제적 및 성능 이점을 제공합니다:

- **[우선순위](https://ai.google.dev/gemini-api/docs/priority-inference) (`"priority"`):** 중요한 앱을 위한 밀리초 지연 시간 (비용 75~100% 추가). 트래픽은 완전히 비중단됩니다.
- **[Flex](https://ai.google.dev/gemini-api/docs/flex-inference) (`"flex"`):** 백그라운드 작업을 위한 1~15분 목표 지연 시간 (비용 50% 절감). 완전히 동기적이지만 기회적 컴퓨팅을 사용합니다.

**이 노트북에서 배울 내용:**
1.  우선순위와 Flex 비교
2.  우선순위 티어 사용 방법
3.  Flex 티어 사용 방법
4.  타임아웃 조정 방법
5.  재시도 구현 방법

> **참고:** 추론 티어는 유료 전용 기능입니다. Flex는 유료 티어의 모든 등급에서 사용 가능하며, 우선순위는 Tier 2 및 3에서 사용 가능합니다. 이 노트북은 무료 티어에서는 실행되지 않습니다. (자세한 내용은 [가격 책정](https://ai.google.dev/pricing)을 참조하세요).

# 설정

### SDK 설치

[PyPI](https://github.com/googleapis/python-genai)에서 SDK를 설치하세요.

In [ ]:
%pip install -U -q "google-genai>=1.70.0"  # Minimum version 1.70 for service tiers

### API 키 설정

다음 셀을 실행하려면 API 키가 `GEMINI_API_KEY`라는 이름의 Colab 시크릿에 저장되어 있어야 합니다. API 키가 없거나 Colab 시크릿 생성 방법이 확실하지 않은 경우 예시는 [인증 ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb)을 참조하세요.

In [2]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### SDK 클라이언트 초기화

API 키로 클라이언트를 초기화합니다:

In [5]:
from google import genai
from google.genai import errors, types

client = genai.Client(api_key=GEMINI_API_KEY)

### 모델 선택

대부분의 Gemini 2.5 및 Gemini 3 모델은 추론 티어를 지원합니다. 자세한 내용은 문서를 참조하세요:
- [Flex 지원 모델](https://ai.google.dev/gemini-api/docs/flex-inference#supported-models)
- [우선순위 지원 모델](https://ai.google.dev/gemini-api/docs/priority-inference#supported-models)

In [18]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## 우선순위와 Flex 비교

| 기능 | 표준 | Flex | 우선순위 | 배치 | 캐싱 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **가격** | 정가 | 50% 할인 | 표준보다 75~100% 추가 | 50% 할인 | 90% 할인 + 토큰 저장 비례 |
| **지연 시간** | 초~분 | 분 (1~15분 목표) | 초 | 최대 24시간 | 더 빠른 첫 토큰 시간 |
| **신뢰성** | 높음/중간-높음 | 최선 노력 (중단 가능) | 높음 (비중단) | 높음 (처리량용) | N/A |
| **인터페이스** | 동기 | 동기 | 동기 | 비동기 | 저장 상태 |
| **최적 사용 사례** | 일반 애플리케이션 워크플로우 | 긴급하지 않은 순차 체인 | 프로덕션, 사용자 대상 앱 | 대규모 데이터셋, 오프라인 평가 | 동일 파일에 대한 반복 쿼리 |

### 우선순위의 주요 이점

* **낮은 지연 시간**: 인터랙티브한 사용자 대상 AI 도구를 위한 초 단위 응답 시간으로 설계되었습니다.
* **높은 신뢰성**: 트래픽이 최고 우선순위로 처리되며 완전히 비중단됩니다.
* **점진적 성능 저하**: 동적 한도를 초과하는 트래픽 급증은 서비스 중단 없이 자동으로 표준 티어로 다운그레이드됩니다.

### Flex의 주요 이점

* **비용 효율성**: 비프로덕션 평가, 백그라운드 에이전트, 데이터 보강에 대한 상당한 비용 절감.
* **낮은 마찰**: 배치 객체, 작업 ID, 폴링을 관리할 필요 없이 기존 요청에 단일 매개변수만 추가하면 됩니다.
* **동기 워크플로우**: 다음 요청이 이전 요청의 출력에 의존하는 순차적 API 체인에 이상적이며, 에이전트 워크플로우에서 배치보다 더 유연합니다.

## 우선순위 추론

Gemini 우선순위 API는 낮은 지연 시간과 최고 신뢰성을 프리미엄 가격으로 요구하는 비즈니스 크리티컬 워크로드를 위한 프리미엄 추론 티어입니다. 우선순위 티어 트래픽은 표준 API 및 Flex 티어 트래픽보다 우선시됩니다.

### 우선순위 작동 방식

우선순위 추론은 요청을 높은 중요도 컴퓨팅 큐로 라우팅하여 사용자 대상 애플리케이션에 예측 가능하고 빠른 성능을 제공합니다. 주요 메커니즘은 동적 한도를 초과하는 트래픽에 대한 서버 측 표준 처리로의 점진적 다운그레이드로, 요청 실패 대신 애플리케이션 안정성을 보장합니다.

### 우선순위 속도 제한

우선순위 소비는 전체 인터랙티브 트래픽 속도 제한에 포함되더라도 자체 속도 제한을 유지합니다. **기본 속도 제한은 각 모델 및 티어에 대한 [표준 속도 제한](https://aistudio.google.com/rate-limit)의 0.3배입니다**.

### 우선순위 사용 방법

우선순위 티어를 사용하려면 요청 본문의 `service_tier` 필드를 `"priority"`로 설정하세요. 필드가 생략된 경우 기본 티어는 표준입니다.

In [19]:
try:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="Why is the sky blue?",
        config={'service_tier': 'priority'},
    )

    # Validate for graceful downgrade
    if response.sdk_http_response.headers.get('x-gemini-service-tier') == "standard":
        print("Warning: Priority limit exceeded, processed at Standard tier.")

    print(response.text)

except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"Error during API call: {e}")

The sky appears blue because of a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white to us, it is actually composed of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength:
*   **Red light** has long, lazy wavelengths.
*   **Blue and violet light** have short, choppy wavelengths.

### 2. The atmosphere acts like an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen) and tiny particles. When sunlight hits the atmosphere, it strikes these gas molecules.

### 3. Scattering occurs
Because blue light travels in shorter, smaller waves, it crashes into the gas molecules more frequently than the longer red waves do. When the blue light hits these molecules, it gets scattered in every direction. 

As you look up, your eyes detect this scattered blue li

응답 헤더에서 서비스 티어를 확인할 수 있습니다:

In [20]:
print(response.sdk_http_response.headers.get('x-gemini-service-tier'))

priority


## Flex 추론

Gemini Flex API는 실시간 성능이 필요하지 않은 지연 시간 허용 워크로드를 위해 동기 처리가 필요하지만 변동 지연 시간과 최선 노력 가용성을 대가로 표준 요금 대비 50% 비용 절감을 제공하는 추론 티어입니다.

### Flex 작동 방식

Gemini Flex 추론은 표준 API와 [배치 API](https://ai.google.dev/gemini-api/docs/batch-api)의 24시간 처리 사이의 간격을 메웁니다. 백그라운드 작업과 순차 워크플로우를 위한 비용 효율적인 솔루션을 제공하기 위해 오프피크 "중단 가능한" 컴퓨팅 용량을 활용합니다.

### Flex 사용 방법

Flex 티어를 사용하려면 요청 본문에 `service_tier`를 `"flex"`로 지정하세요. 기본적으로 이 필드가 생략된 경우 요청은 표준 티어를 사용합니다.

In [21]:
try:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="Why is the sky blue?",
        config={'service_tier': 'flex'},
    )

    print(response.text)
except errors.APIError as e:
    print(e.code, e.message)

The short answer is that the sky is blue because of a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of why it happens:

### 1. Sunlight is made of all colors
Although sunlight looks white to us, it is actually composed of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength:
*   **Red light** has long, lazy wavelengths.
*   **Blue and violet light** have short, choppy wavelengths.

### 2. The Atmosphere is an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen). As sunlight reaches the atmosphere, it strikes the molecules of these gases and scatters in every direction.

### 3. Blue light scatters the most
Because blue light travels in shorter, smaller waves, it hits the gas molecules more frequently and is scattered much more strongly than the other colors. This "scattered" blue light is what your eyes see coming from every 

In [22]:
print(response.sdk_http_response.headers.get('x-gemini-service-tier'))

flex


### 타임아웃 설정 조정

REST API 및 클라이언트 라이브러리에 대한 요청별 타임아웃을 구성할 수 있으며, 클라이언트 라이브러리를 사용할 때는 전역 타임아웃도 설정 가능합니다.

클라이언트 측 타임아웃이 의도된 서버 대기 시간(예: Flex 대기 큐에 대해 600초 이상)을 커버하도록 항상 확인하세요. SDK는 밀리초 단위의 타임아웃 값을 기대합니다.

#### 요청별 타임아웃

In [24]:
try:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents="why is the sky blue?",
        config={
            "service_tier": "flex",
            "http_options": {"timeout": 900000}
        },
    )
    print(response.text)
except errors.APIError as e:
    print(e.code, e.message)

The sky appears blue due to a phenomenon called **Rayleigh scattering**. 

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white to us, it is actually made up of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength:
*   **Red light** has long, lazy, stretched-out wavelengths.
*   **Blue and violet light** have short, choppy, frequent wavelengths.

### 2. The Atmosphere acts like an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen). When sunlight hits the atmosphere, it collides with these gas molecules.

### 3. Scattering happens
Because blue light travels in shorter, smaller waves, it is much more likely to strike these gas molecules and get **scattered** in every direction. Red and yellow light, with their longer wavelengths, pass through the atmosphere mostly uninterrupted.

So, when 

스트리밍을 사용한 예시 (타임아웃은 구현에 따라 각 청크 또는 전체에 적용됨)

In [25]:
try:
    response = client.models.generate_content_stream(
        model=MODEL_ID,
        contents=["List 5 ideas for a sci-fi movie."],
        config={
            "service_tier": "flex",
            "http_options": {"timeout": 60000}
        },
        # Per-request timeout for the streaming operation
    )
    for chunk in response:
        print(chunk.text, end="")
except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"An error occurred during streaming: {e}")

Here are five original sci-fi movie concepts, ranging from psychological thrillers to high-concept space operas:

### 1. The Echo Archive
**The Concept:** In a future where memories can be digitised and "donated" to a collective global library, a professional "Memory Editor" discovers a series of memories that shouldn’t exist. They depict a secret history of the world that conflicts with official records. 
**The Hook:** To uncover the truth, the protagonist must navigate the fractured consciousness of strangers, but they soon realize that every time they access a forbidden memory, a piece of their own personality is permanently overwritten.

### 2. Orbital Decay
**The Concept:** Earth has become a prison planet, entirely encased in a sophisticated, automated satellite grid that prevents anything from leaving or entering. A low-level technician tasked with maintaining the "Kill-Switch" satellites discovers that the grid isn't keeping the world's threats *in*—it’s protecting the planet f

#### 전역 타임아웃

특정 `genai.Client` 인스턴스(클라이언트 라이브러리만 해당)를 통해 이루어진 모든 API 호출에 기본 타임아웃을 갖게 하려면 `http_options`와 `genai.types.HttpOptions`를 사용하여 클라이언트 초기화 시 구성할 수 있습니다.

In [27]:
global_timeout_ms = 120000

client_with_global_timeout = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(timeout=global_timeout_ms)
)

try:
    # Calling generate_content using global timeout...
    response = client_with_global_timeout.models.generate_content(
        model=MODEL_ID,
        contents="Summarize the history of AI development since 2000.",
        config={"service_tier": "flex"},
    )
    print(response.text)

    # A per-request timeout will *override* the global timeout for that specific call.
    shorter_timeout = 30000
    response = client_with_global_timeout.models.generate_content(
        model=MODEL_ID,
        contents="Provide a very brief definition of machine learning.",
        config={
            "service_tier": "flex",
            "http_options":{"timeout": shorter_timeout}
        }  # Overrides the global timeout
    )

    print(response.text)

except TimeoutError:
    print(
        f"A GenerateContent call timed out. Check if the global or per-request timeout was exceeded."
    )
except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"An error occurred: {e}")

The history of AI since 2000 is a transition from symbolic, rule-based systems to probabilistic, data-driven "deep learning." This evolution can be broken down into four distinct eras.

### 1. The Era of Foundation and Narrow AI (2000–2010)
At the turn of the millennium, AI research was focused on specialized tasks rather than general intelligence.
*   **The Rise of Machine Learning:** Researchers moved away from "expert systems" (where humans hard-coded rules) toward systems that learned from data. Statistical methods, such as Support Vector Machines (SVMs), dominated.
*   **Milestones:** In 2005, a Stanford robot won the DARPA Grand Challenge by navigating a desert course, proving that autonomous navigation was possible. 
*   **The Big Data Catalyst:** The explosion of the internet began providing the massive datasets required to train more sophisticated models.

### 2. The Deep Learning Breakthrough (2011–2015)
This period marked the most significant shift in AI history, triggered b

### 재시도 구현

Flex는 중단 가능하며 503 오류로 실패하므로, 다음은 실패한 요청을 계속 처리하기 위한 선택적 재시도 로직 구현 예시입니다:

In [29]:
import time


def call_with_retry(max_retries=3, base_delay=5):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_ID,
                contents="Provide a very brief definition of machine learning.",
                config={"service_tier": "flex"},
            )
        except errors.APIError as e:
            # Check for 503 Service Unavailable or 429 Rate Limits
            print(e.code)
            if attempt < max_retries - 1:
                delay = base_delay * (2**attempt)  # Exponential Backoff
                print(f"Flex busy, retrying in {delay}s...")
                time.sleep(delay)
            else:
                # Fallback to standard on last strike
                print("Flex exhausted, falling back to Standard...")
                return client.models.generate_content(
                    model=MODEL_ID,
                    contents="Provide a very brief definition of machine learning.",
                )
        except Exception as e:
            print(f"An error occurred: {e}")

# Usage
response = call_with_retry()
print(response.text)


Machine learning is a field of artificial intelligence that enables computers to learn from data and improve at tasks without being explicitly programmed for them.


## 추가 리소스

자세한 내용은 다음 리소스를 참조하세요:

- [최적화 문서](https://ai.google.dev/gemini-api/docs/optimization)
- [우선순위 문서](https://ai.google.dev/gemini-api/docs/priority-inference)
- [Flex 문서](https://ai.google.dev/gemini-api/docs/flex-inference)